# Contribution of parameters via ablation study

This notebook studies the importance of each parameter in the ability to predict the encoding and decoding energy.
The idea is to assess the accuracy of energy prediction using a model that takes all parameters as inputs except the one whose influence we wish to measure.

In [1]:
from mendevi.models.nn import DNN  # better than LR and GPR

In [2]:
dataset = "scatter.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation

## Reference model, with all input parameters without ommision

The idea here is to have the best possible model to use as a reference for comparing the decline of other models.

In [3]:
# define the inputs and the models properties
encode_inputs = [
    ("encoder",),
    ("effort",),
    ("mode",),
    ("quality",),
    ("hostname", "temperature"),
    ("nb_frames",),
    ("pixels",),
    ("framerate",),
    ("color_primaries", "color_transfer"),
    ("spatial_dct", "temporal_dct", "rms_sobel", "rms_time_diff"),
]
decode_inputs = [
    ("decoder",),
    ("pixels",),
    ("framerate",),
    ("hostname", "temperature"),
    ("color_primaries", "color_transfer"),
]

niter = 15000  # number of training iterations, default 10000
learning_rate = 2e-5  # adam gradient decay learning rate, default 1e-4
total = 25_000  # number of model parameters, default based on number of observations

In [4]:
# model training
encode_reference = DNN(
    weights="encode_ablation_study",  # optional, for checkpoint (in ~/.cache/mendevi)
    input_labels=[*lbls for lbls in encode_inputs],
    aggregation=[],  # optional, this feature force a single common model
    output_labels=["log_act_duration_per_frame", "power"],
    niter=niter, learning_rate=learning_rate, total=total,
)
decode_reference = DNN(
    weights="decode_ablation_study",  # optional, for checkpoint (in ~/.cache/mendevi)
    input_labels=[*lbls for lbls in decode_inputs],
    aggregation=[],  # optional, this feature force a single common model
    output_labels=["log_act_duration_per_frame", "power"],
    niter=niter, learning_rate=learning_rate, total=total,
)
if not encode_reference.is_fit():
    encode_reference.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val}",
)
if not decode_reference.is_fit():
    decode_reference.fit(
        dataset,
        table="t_dec_decode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val}",
)

Fit 'regressive model to predict log_act_duration_per_frame, power from color_primaries, color_transfer, effort, encoder, framerate, hostname, mode, nb_frames, pixels, quality, rms_sobel, rms_time_diff, spatial_dct, temperature, temporal_dct'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 28562 observations
█ █ NotImplementedError l693 (The one hot encoding for 'hostname' is not defined.)
█ NotImplementedError l693 (The one hot encoding for 'hostname' is not defined.)


NotImplementedError: The one hot encoding for 'hostname' is not defined.

In [ ]:
# validation from dataset
encode_reference.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")
decode_reference.validate(dataset, table="t_dec_decode", select=f"uniform(vid_hash) < {val}")

In [ ]:
# prediction from a video
pred = dnn_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)